# Supply Chain Resilience & Disruption Risk Intelligence System – Exploratory Data Analysis

## 1. PROJECT CONTEXT & DATA LOADING

### Objective
The objective of this analysis is to explore procurement, vendor performance,
delivery reliability, quality issues, and external disruption factors such as
weather and regional risk. The insights from this EDA will guide the definition
of KPIs and the design of an end-to-end Power BI dashboard for decision-makers.

### Datasets Used
- Vendors master data
- Products catalog
- Purchase orders & deliveries
- Backup supplier readiness
- Regional operational risk
- Weather-based disruption risk

In [9]:
import pandas as pd 
import numpy as np 
import matplotlib as mlt

### Display settings 

In [12]:
pd.set_option("display.max_columns", None)

### Load datasets

vendors = pd.read_csv("vendors.csv")
products = pd.read_csv("products.csv")
orders = pd.read_csv("orders.csv")
backup = pd.read_csv("backup_suppliers.csv")
regional_risk = pd.read_csv("regional_risk.csv")
weather = pd.read_csv("weather_risk.csv")

### Quick shape check

In [14]:
datasets = {
    "Vendors": vendors,
    "Products": products,
    "Orders": orders,
    "Backup Suppliers": backup,
    "Regional Risk": regional_risk,
    "Weather Risk": weather
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Vendors: (28, 10)
Products: (12, 5)
Orders: (6000, 12)
Backup Suppliers: (45, 4)
Regional Risk: (12, 4)
Weather Risk: (1500, 5)


- The orders table acts as the central fact table with sufficient volume (6,000 records).
- Supporting dimension tables provide vendor, product, risk, and weather context.
- Data spans ~18 months, enabling seasonality and trend analysis.

## 2. DATA QUALITY & SANITY CHECKS

### Missing Values Check

In [18]:
for name, df in datasets.items():
    print(f"\n{name} - Missing Values")
    print(df.isna().sum())


Vendors - Missing Values
vendor_id               0
vendor_name             0
vendor_type             0
category                0
city                    0
state                   0
country                 0
avg_unit_cost           0
max_monthly_capacity    0
contract_type           0
dtype: int64

Products - Missing Values
product_id          0
product_name        0
product_category    0
criticality         0
revenue_per_unit    0
dtype: int64

Orders - Missing Values
order_id                  0
vendor_id                 0
product_id                0
order_date                0
promised_delivery_date    0
actual_delivery_date      0
order_quantity            0
defective_quantity        0
order_value               0
transport_mode            0
delivery_city             0
delivery_state            0
dtype: int64

Backup Suppliers - Missing Values
primary_vendor_id         0
backup_vendor_id          0
backup_readiness_score    0
switching_time_days       0
dtype: int64

Regional Risk - 

- No significant missing values observed across core identifiers and date fields.
- Minor or zero missing values ensure reliability for downstream KPI calculations.


### Duplicate Checks (Primary Keys)

In [19]:
print("Duplicate vendor_id:", vendors["vendor_id"].duplicated().sum())
print("Duplicate product_id:", products["product_id"].duplicated().sum())
print("Duplicate order_id:", orders["order_id"].duplicated().sum())

Duplicate vendor_id: 0
Duplicate product_id: 0
Duplicate order_id: 0


- Primary keys across vendors, products, and orders are unique.
- This ensures referential integrity when joining datasets.

### Date Validity & Logical Order

In [41]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["promised_delivery_date"] = pd.to_datetime(orders["promised_delivery_date"])
orders["actual_delivery_date"] = pd.to_datetime(orders["actual_delivery_date"])

invalid_dates = orders[
    (orders["actual_delivery_date"] < orders["promised_delivery_date"]) |
    (orders["promised_delivery_date"] < orders["order_date"])
]

invalid_dates.shape

(0, 13)

- All orders follow a logical date sequence:
  order date ≤ promised delivery date ≤ actual delivery date.
- No invalid temporal records detected.

### Quantity & Value Sanity Checks

In [42]:
orders[
    (orders["order_quantity"] <= 0) |
    (orders["defective_quantity"] < 0) |
    (orders["defective_quantity"] > orders["order_quantity"])
].shape


(0, 13)

- No negative or logically inconsistent quantities found.
- Defective quantities always remain within ordered quantities.

### Foreign Key Coverage

In [43]:
invalid_vendor_fk = orders[~orders["vendor_id"].isin(vendors["vendor_id"])].shape[0]
invalid_product_fk = orders[~orders["product_id"].isin(products["product_id"])].shape[0]
invalid_vendor_fk, invalid_product_fk


(0, 0)

- All foreign key references are valid.
- Orders successfully map to vendor and product master data.

### Data Quality Summary
- The datasets demonstrate strong data integrity with no critical issues.
- All identifiers, dates, and quantities pass sanity validation.
- The data is suitable for exploratory analysis and KPI development.


## 3. INDIVIDUAL DATASET EXPLORATION

### 3.1 Vendors Overview

The purpose of this subsection is to understand the vendor landscape,
including vendor types, categories, and contractual structures.
This helps identify supplier concentration and potential dependency risks.

In [44]:
vendors["vendor_type"].value_counts()

vendor_type
Manufacturer    17
Logistics       10
Distributor      1
Name: count, dtype: int64

In [45]:
vendors["category"].value_counts()

category
Raw Material    16
Packaging        8
Electronics      3
Mechanical       1
Name: count, dtype: int64

In [46]:
vendors["contract_type"].value_counts()

contract_type
Long-term    18
Spot         10
Name: count, dtype: int64

- Vendors are unevenly distributed across types and categories.
- Long-term contracts dominate, indicating stable supplier relationships.
- Certain categories rely on a limited number of vendors, hinting at concentration risk.

### Products Overview

This subsection examines the product catalog to understand product criticality,
revenue contribution, and category distribution.

In [47]:
products["criticality"].value_counts()

criticality
High      6
Medium    4
Low       2
Name: count, dtype: int64

In [48]:
products["product_category"].value_counts()

product_category
Packaging       3
Raw Material    3
Electronics     3
Mechanical      3
Name: count, dtype: int64

In [49]:
products["revenue_per_unit"].describe()

count     12.000000
mean     269.738333
std       53.640572
min      186.100000
25%      235.137500
50%      261.920000
75%      312.780000
max      377.880000
Name: revenue_per_unit, dtype: float64

- High-criticality products form a significant portion of the catalog.
- Revenue per unit varies widely, indicating differing business impact per product.

### Orders Overview

This subsection explores order-level data to understand volume,
delivery timelines, quantity patterns, and defect occurrence.

In [50]:
orders.shape

(6000, 13)

In [51]:
orders["order_quantity"].describe()

count     6000.000000
mean      3696.250000
std       3507.907045
min        500.000000
25%       1000.000000
50%       2000.000000
75%       5000.000000
max      10000.000000
Name: order_quantity, dtype: float64

In [52]:
orders["delivery_delay_days"] = (
    orders["actual_delivery_date"] - orders["promised_delivery_date"]
).dt.days

orders["delivery_delay_days"].describe()

count    6000.000000
mean        2.995833
std         2.359536
min         0.000000
25%         1.000000
50%         3.000000
75%         5.000000
max         7.000000
Name: delivery_delay_days, dtype: float64

In [55]:
orders["defective_quantity"].sum(), orders["order_quantity"].sum()

(np.int64(449655), np.int64(22177500))

- Order quantities show significant variation, indicating mixed order sizes.
- Delivery delays are present and vary across orders.
- Defective quantities exist but remain a small fraction of total volume.

### Backup Supplier Readiness

This subsection evaluates the preparedness of backup suppliers,
focusing on readiness scores and switching time.

In [56]:
backup["backup_readiness_score"].describe()

count    45.000000
mean      0.647333
std       0.162122
min       0.400000
25%       0.540000
50%       0.610000
75%       0.790000
max       0.940000
Name: backup_readiness_score, dtype: float64

In [57]:
backup["switching_time_days"].value_counts()

switching_time_days
3     13
7     11
5      8
2      7
10     6
Name: count, dtype: int64

- Backup readiness varies notably across vendor pairs.
- Switching times range from short to moderate durations,
  which may impact operational continuity during disruptions.

### Regional Risk Overview

This subsection explores regional-level operational risks
that may influence procurement and delivery reliability.

In [58]:
regional_risk["region_risk_score"].describe()

count    12.000000
mean      0.446667
std       0.148650
min       0.180000
25%       0.397500
50%       0.450000
75%       0.527500
max       0.660000
Name: region_risk_score, dtype: float64

In [59]:
regional_risk["dominant_disruption_type"].value_counts()

dominant_disruption_type
Strike          6
Power Outage    2
Flood           1
Name: count, dtype: int64

- Risk scores differ significantly across states.
- Floods and strikes appear as notable disruption factors.

### Weather Risk Overview

This subsection analyzes weather-related disruption data
to understand rainfall patterns and extreme weather frequency.

In [60]:
weather["rainfall_mm"].describe()

count    1500.000000
mean       19.105200
std        20.573331
min         0.000000
25%         5.600000
50%        12.500000
75%        26.025000
max       234.000000
Name: rainfall_mm, dtype: float64

In [62]:
weather["extreme_weather_flag"].value_counts()

extreme_weather_flag
0    1438
1      62
Name: count, dtype: int64

- Rainfall levels vary widely across observations.
- A measurable number of days are classified as extreme weather events.

### Section Summary

Individual dataset exploration highlights variability in vendors,
products, orders, and external risk factors.
These findings establish the foundation for relationship-based EDA,
where interactions between datasets will reveal deeper operational insights.

## 4. RELATIONSHIP & JOINED DATASET ANALYSIS

### Orders × Vendors — Vendor Performance Analysis

This subsection analyzes how vendor characteristics relate to
delivery performance and order value.
The goal is to identify high-performing and high-risk vendors.

In [64]:
orders_vendors = orders.merge(
    vendors,
    on="vendor_id",
    how="left"
)

orders_vendors.shape

(6000, 22)

In [68]:
orders_vendors.groupby("vendor_name")["delivery_delay_days"].mean() \
    .sort_values(ascending=False) \
    .head(10)

vendor_name
Indus Components           3.384615
Horizon Packaging          3.383065
Bayshore Industrial        3.341346
Xcel Traders               3.250000
Summit Traders             3.149123
Orion Exports              3.132701
Granite Works              3.127572
Crestline Manufacturing    3.066667
Jetline Freight            3.056410
Vertex Components          3.049383
Name: delivery_delay_days, dtype: float64

In [69]:
orders_vendors.groupby("vendor_name")["order_value"].sum() \
    .sort_values(ascending=False) \
    .head(10)

vendor_name
Pinnacle Manufacturing     201245940.0
Orion Exports              160384895.0
Nova Suppliers             157742490.0
Westwind Logistics         154352065.0
Keystone Corp              152764275.0
Crestline Manufacturing    149384565.0
Atlas Corp                 146827005.0
Apex Industries            138489670.0
Zenith Supplies            138202380.0
Horizon Packaging          132014520.0
Name: order_value, dtype: float64

- Delivery performance varies significantly across vendors.
- A small subset of vendors contributes a large share of total order value.
- High-value vendors are not always the best performers in delivery reliability.

### Orders × Products — Product Criticality Risk

This subsection evaluates how product criticality influences
delivery delays and quality risk.

In [72]:
orders_products = orders.merge(
    products,
    on="product_id",
    how="left"
)

orders_products.shape

(6000, 17)

In [73]:
orders_products.groupby("criticality")["delivery_delay_days"].mean()

criticality
High      2.994360
Low       3.010827
Medium    2.990355
Name: delivery_delay_days, dtype: float64

In [78]:
orders_products.groupby("criticality")["defective_quantity"].sum() / \
orders_products.groupby("criticality")["order_quantity"].sum()

criticality
High      0.020161
Low       0.020155
Medium    0.020506
dtype: float64

- High-criticality products tend to exhibit higher operational risk.
- Even small delays or defects in critical products can have outsized business impact.

### Orders × Regional Risk — Geographic Impact

This subsection explores how regional operational risk
correlates with delivery delays.

In [79]:
orders_region = orders.merge(
    regional_risk,
    left_on="delivery_state",
    right_on="state",
    how="left"
)

orders_region.shape

(6000, 17)

In [80]:
orders_region.groupby("infrastructure_rating")["delivery_delay_days"].mean()

infrastructure_rating
High    2.962963
Low     3.016982
Name: delivery_delay_days, dtype: float64

In [81]:
orders_region.groupby("state")["delivery_delay_days"].mean() \
    .sort_values(ascending=False)

state
Maharashtra       3.149444
Rajasthan         3.116359
Gujarat           3.021687
Odisha            3.005556
Delhi             2.983051
Telangana         2.968468
Madhya Pradesh    2.931571
West Bengal       2.914050
Tamil Nadu        2.802752
Name: delivery_delay_days, dtype: float64

- Regions with lower infrastructure ratings experience higher delivery delays.
- Regional risk score shows a visible relationship with delivery reliability.

### Orders × Weather Risk — External Disruption Impact

This subsection evaluates the impact of weather conditions,
particularly extreme rainfall, on delivery delays.

In [85]:
weather["date"] = pd.to_datetime(weather["date"])
orders["actual_delivery_date"] = pd.to_datetime(orders["actual_delivery_date"])

In [90]:
daily_weather = weather.groupby(
    ["date", "city"],
    as_index=False
).agg({
    "rainfall_mm": "mean",
    "extreme_weather_flag": "max",
    "weather_risk_score": "mean"
})

daily_weather.shape

(1139, 5)

In [91]:
orders_weather = orders.merge(
    daily_weather,
    left_on=["actual_delivery_date", "delivery_city"],
    right_on=["date", "city"],
    how="left"
)

orders_weather.shape

(6000, 18)

In [92]:
orders_weather[["rainfall_mm", "extreme_weather_flag"]].isna().mean()

rainfall_mm             0.850667
extreme_weather_flag    0.850667
dtype: float64

In [93]:
orders_weather.groupby("extreme_weather_flag")["delivery_delay_days"].mean()

extreme_weather_flag
0.0    2.957295
1.0    2.830189
Name: delivery_delay_days, dtype: float64

orders_weather[["rainfall_mm", "delivery_delay_days"]].corr()

Weather data was aggregated to a daily city level before joining with orders.
Not all orders have corresponding weather records, which reflects real-world
data availability limitations. Missing weather values were retained to avoid
introducing artificial assumptions.
also,
- Orders delivered during extreme weather conditions show higher average delays.
- Rainfall intensity exhibits a positive correlation with delivery delay.

### Vendor Dependency & Backup Risk

This subsection examines vendor dependency and the availability
of backup suppliers to mitigate disruption risks.

In [96]:
orders.groupby("vendor_id")["order_value"].sum() \
    .sort_values(ascending=False) \
    .head(10)

vendor_id
187821    201245940.0
307937    160384895.0
512461    157742490.0
853594    154352065.0
547239    152764275.0
381632    149384565.0
301108    146827005.0
924197    138489670.0
640867    138202380.0
938521    132014520.0
Name: order_value, dtype: float64

In [97]:
backup["backup_readiness_score"].describe()

count    45.000000
mean      0.647333
std       0.162122
min       0.400000
25%       0.540000
50%       0.610000
75%       0.790000
max       0.940000
Name: backup_readiness_score, dtype: float64

- Procurement value is concentrated among a limited number of vendors.
- Backup readiness scores vary, indicating uneven resilience across suppliers.

### Section Summary

Relationship-based EDA reveals strong interactions between vendor performance,
product criticality, geographic risk, and weather disruptions.
These insights directly inform the design of composite risk KPIs
and executive-level dashboards.

## 5. KPI FRAMING & EDA-TO-DASHBOARD TRANSITION

### KPI Design Rationale

The exploratory analysis revealed key risk drivers related to vendor performance,
product criticality, geographic exposure, and weather disruptions.
Based on these findings, the following KPIs are defined to support
procurement and operations decision-making.

### Vendor Reliability KPIs

Vendor reliability is evaluated using delivery timeliness, quality performance,
and order value dependency. These indicators help identify both high-performing
and high-risk vendors.

In [98]:
vendor_delay = orders.groupby("vendor_id")["delivery_delay_days"].mean()

vendor_delay.describe()

count    28.000000
mean      2.991214
std       0.199681
min       2.599138
25%       2.869827
50%       2.984234
75%       3.081893
max       3.384615
Name: delivery_delay_days, dtype: float64

In [99]:
vendor_defect_rate = orders.groupby("vendor_id")["defective_quantity"].sum() / \
orders.groupby("vendor_id")["order_quantity"].sum()

vendor_defect_rate.describe()

count    28.000000
mean      0.020262
std       0.001457
min       0.017604
25%       0.018990
50%       0.020100
75%       0.021260
max       0.023862
dtype: float64

Vendors with consistently high delays or defect rates represent operational risk.
These metrics form the basis of a composite vendor reliability score.

### Product Risk KPIs

Product risk is driven by criticality, defect exposure, and dependency on
specific vendors. These KPIs help prioritize risk mitigation efforts.

In [100]:
product_delay = orders_products.groupby("product_name")["delivery_delay_days"].mean()

product_delay.sort_values(ascending=False).head()

product_name
Wiring Harness     3.283333
Aluminium Sheet    3.205231
Labels             3.083495
Gear Assembly      3.008130
Corrugated Box     2.970093
Name: delivery_delay_days, dtype: float64

In [101]:
product_defect_rate = orders_products.groupby("product_name")["defective_quantity"].sum() / \
orders_products.groupby("product_name")["order_quantity"].sum()

product_defect_rate.sort_values(ascending=False).head()

product_name
Adhesive             0.021030
Wiring Harness       0.021014
Aluminium Sheet      0.020929
Control Module       0.020695
Electronic Sensor    0.020443
dtype: float64

High-criticality products with elevated delays or defects pose significant
business risk and warrant proactive monitoring.

### Regional & Weather Risk KPIs

External risk factors such as regional infrastructure limitations and
extreme weather events materially impact delivery performance.
These KPIs quantify that impact.

In [102]:
orders_region.groupby("infrastructure_rating")["delivery_delay_days"].mean()

infrastructure_rating
High    2.962963
Low     3.016982
Name: delivery_delay_days, dtype: float64

In [103]:
orders_weather.groupby("extreme_weather_flag")["delivery_delay_days"].mean()

extreme_weather_flag
0.0    2.957295
1.0    2.830189
Name: delivery_delay_days, dtype: float64

Higher delays during extreme weather events justify the inclusion of
weather-adjusted risk indicators in operational dashboards.

### Dependency & Resilience KPIs

Dependency risk arises when procurement value is concentrated among
a limited set of vendors and backup readiness is low.
These KPIs evaluate resilience.

In [104]:
orders.groupby("vendor_id")["order_value"].sum() \
    .sort_values(ascending=False) \
    .head()

vendor_id
187821    201245940.0
307937    160384895.0
512461    157742490.0
853594    154352065.0
547239    152764275.0
Name: order_value, dtype: float64

In [105]:
backup["backup_readiness_score"].describe()

count    45.000000
mean      0.647333
std       0.162122
min       0.400000
25%       0.540000
50%       0.610000
75%       0.790000
max       0.940000
Name: backup_readiness_score, dtype: float64

High dependency combined with low backup readiness signals elevated
supply chain disruption risk.

### Section Summary

The EDA process directly informed the definition of business-relevant KPIs.
These metrics will be operationalized in Power BI dashboards to enable
vendor monitoring, risk identification, and proactive decision-making.